# 1 · Profiling the definition space

A vulnerability scanner reports findings under *plugins*, and a plugin's entire
meaning lives in its title. Before writing a single rule, the question worth
answering is: **what kinds of titles are there, and does each kind behave the
same way?**

That turned out to matter more than any individual rule. Supersedence is
decidable for one kind of title and provably undecidable for another, and no
amount of rule tuning moves that.

> The data here is synthetic (`src/synthetic.py`) and generated to reproduce the
> *structure* of a real export — not its content. No client data is in this repo.

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
pd.set_option("display.width", 110)
pd.set_option("display.max_columns", 20)

from src.synthetic import generate
from src.rules import frame_from_names

raw = generate()
findings = frame_from_names(raw)

print(f"findings          {len(findings):,}")
print(f"assets            {findings['asset'].nunique():,}")
print(f"distinct plugins  {findings['definition_name'].nunique():,}")
print(f"base rate of SUPERSEDED  "
      f"{findings['analyst_category'].eq('SUPERSEDED').mean():.3f}")

findings          3,640
assets            891
distinct plugins  18
base rate of SUPERSEDED  0.113


## The taxonomy

Every title falls into one of a few syntactic shapes. Grouping by shape and
looking at how often the analyst calls each one *superseded* is the cheapest
possible orientation — one `groupby` that decides where to spend the next week.

In [ ]:
profile = (findings.groupby("shape")
    .agg(findings=("finding_id", "size"),
         plugins=("definition_name", "nunique"),
         superseded=("analyst_category", lambda s: s.eq("SUPERSEDED").sum()))
    .assign(rate=lambda d: (d["superseded"] / d["findings"]).round(3))
    .sort_values("findings", ascending=False))
print(profile.to_string())

            findings  plugins  superseded   rate
shape                                           
comparator      2188        8         295  0.135
kb_titled        750        3          58  0.077
protocol         440        2          38  0.086
monthly          132        2          10  0.076
eol               95        2           7  0.074
config            35        1           5  0.143


## The precondition nobody states

A comparison rule can only see supersedence when **two branches of the same
product are present in the same export**. That happens because the scanner keeps
reporting findings under the old plugin until every machine catches up.

If a product has only one branch in the cut, there is nothing to compare against
— and no rule over that axis can ever fire, however well written.

This is a *structural* ceiling, not a modelling one, so it is worth measuring
explicitly rather than discovering later as unexplained recall.

In [ ]:
branches = (findings[findings["shape"].eq("comparator")]
    .groupby("product")
    .agg(findings=("finding_id", "size"),
         branches=("major", "nunique"),
         which=("major", lambda s: sorted(s.dropna().unique().tolist())),
         superseded=("analyst_category", lambda s: s.eq("SUPERSEDED").sum()))
    .sort_values("findings", ascending=False))
print(branches.to_string())

decidable = branches["branches"] > 1
print(f"\nproducts a comparison rule can decide: {int(decidable.sum())} of {len(branches)}")
print(f"findings inside them:                  {int(branches.loc[decidable, 'findings'].sum()):,}")
print(f"findings structurally out of reach:    {int(branches.loc[~decidable, 'findings'].sum()):,}")

                            findings  branches            which  superseded
product                                                                    
contoso browser (chromium)      2040         3  [149, 150, 151]         282
fabrikam node.js 22.x             60         1             [22]           4
northwind reader                  48         2         [11, 26]           7
apache log4j seol (               40         1              [1]           2

products a comparison rule can decide: 2 of 4
findings inside them:                  2,088
findings structurally out of reach:    100


## What this buys

Two facts, both of which shaped everything downstream:

1. Only the **comparator** shape carries a usable signal. The others sit at or
   near the base rate, which is the profiling equivalent of a flat line.
2. Even inside that shape, the reachable population is bounded by how many
   products happen to have two live branches — a property of the fleet, not of
   the code.

Next: turn that into a rule, and resist the obvious version of it.